Import Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
file_path = "Rata-rata gaji tahunan per sektor 2015-2025 (1).xlsx"

df = pd.read_excel(file_path)

df.head()

In [ ]:
print(df.shape)
print(df.info())
print(df.isnull().sum())

Cleaning

In [ ]:
new_columns = ['Sektor'] + df.iloc[0, 1:].tolist()

df.columns = new_columns

df = df.drop(0).reset_index(drop=True)

df.head()

In [ ]:
df = df.dropna(subset=['Sektor'])

In [ ]:
numeric_cols = df.columns[1:]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

Missing Value

In [ ]:
df[numeric_cols] = df[numeric_cols].fillna(
    df[numeric_cols].median()
)

In [ ]:
df.isnull().sum()

Penanganan

In [ ]:
df[numeric_cols] = df[numeric_cols].fillna(
    df[numeric_cols].median()
)

Penanganan Outlier

In [ ]:
plt.figure(figsize=(12,5))

sns.boxplot(data=df[numeric_cols])

plt.title("Boxplot Sebelum Penanganan Outlier")
plt.xticks(rotation=45)

plt.show()

In [ ]:
winsor_df = raw_df.copy()

for col in numeric_cols:
    Q1 = winsor_df[col].quantile(0.25)
    Q3 = winsor_df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    winsor_df[col] = np.where(
        winsor_df[col] < lower,
        lower,
        winsor_df[col]
    )

    winsor_df[col] = np.where(
        winsor_df[col] > upper,
        upper,
        winsor_df[col]
    )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))

# Sebelum
sns.boxplot(data=df[numeric_cols], ax=axes[0])
axes[0].set_title("Sebelum Winsorization")

# Sesudah
sns.boxplot(data=winsor_df[numeric_cols], ax=axes[1])
axes[1].set_title("Sesudah Winsorization")

plt.tight_layout()
plt.show()

EDA

In [ ]:
winsor_df.describe()

In [ ]:
avg_salary = winsor_df[numeric_cols].mean()

plt.figure(figsize=(10,5))

plt.plot(
    avg_salary.index,
    avg_salary.values,
    marker='o'
)

plt.title("Rata-rata Gaji per Tahun")
plt.xlabel("Tahun")
plt.ylabel("Gaji")
plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.histplot(
    winsor_df[2025],
    bins=10,
    kde=True
)

plt.title("Distribusi Gaji Tahun 2025")

plt.show()

In [ ]:
plt.figure(figsize=(10,6))

sns.heatmap(
    winsor_df[numeric_cols].corr(),
    annot=True,
    cmap='coolwarm'
)

plt.title("Korelasi Antar Tahun")

plt.show()

In [ ]:
top_salary = winsor_df[['Sektor', 2025]] \
    .sort_values(by=2025, ascending=False)

top_salary.head(10)

In [ ]:
plt.figure(figsize=(10,6))

sns.barplot(
    data=top_salary.head(10),
    x=2025,
    y='Sektor'
)

plt.title("10 Sektor dengan Gaji Tertinggi Tahun 2025")

plt.show()

In [ ]:
winsor_df['Growth_%'] = (
    (winsor_df[2025] - winsor_df[2015])
    / winsor_df[2015]
) * 100

In [ ]:
growth_df = winsor_df[
    ['Sektor', 'Growth_%']
].sort_values(by='Growth_%', ascending=False)

growth_df.head()

In [ ]:
plt.figure(figsize=(10,6))

sns.barplot(
    data=growth_df,
    x='Growth_%',
    y='Sektor'
)

plt.title("Pertumbuhan Gaji 2015-2025")

plt.show()

In [ ]:
long_df = winsor_df.melt(
    id_vars='Sektor',
    var_name='Tahun',
    value_name='Gaji'
)

long_df.head()

In [ ]:
winsor_df.to_csv(
    "gaji_clean_fix.csv",
    index=False
)